In [1]:
# Imports and Configuration

import os
import time
import logging
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Tuple
from pathlib import Path
from collections import deque
from dotenv import load_dotenv
from groq import Groq
import chromadb
from chromadb.utils import embedding_functions

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

MAX_RETRIES = 3
RETRY_DELAY = 2
DEFAULT_MODEL = "llama-3.3-70b-versatile"
DEFAULT_MAX_TOKENS = 1024
DEFAULT_TEMPERATURE = 0.7
SHORT_TERM_LIMIT = 10
MEMORY_RESULTS = 3
CHROMA_PATH = "C:/educational files/advanced_agent/memory/chroma_store"

In [2]:
# Client Initialization

env_path = Path("C:/educational files/advanced_agent/.env")
load_dotenv(dotenv_path=env_path)

def init_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise EnvironmentError("GROQ_API_KEY not found in .env")
    log.info("Groq client initialized successfully")
    return Groq(api_key=api_key)

client = init_client()

2026-06-02 11:19:46,319 [INFO] Groq client initialized successfully


In [3]:
# Agent Configuration

@dataclass
class AgentConfig:
    model: str = DEFAULT_MODEL
    max_tokens: int = DEFAULT_MAX_TOKENS
    temperature: float = DEFAULT_TEMPERATURE
    system_prompt: str = (
        "You are an advanced autonomous AI agent with two-layer memory: "
        "short-term buffer for recent exchanges and long-term vector memory for semantic recall. "
        "When memory context is injected into the prompt, treat it as verified past knowledge. "
        "Prioritize recalled context when answering follow-up questions. "
        "Be concise, precise, and professional."
    )
    session_token_count: int = field(default=0, repr=False)
    memory_hits: int = field(default=0, repr=False)

config = AgentConfig()
log.info(f"Agent configured — model: {config.model} | memory tracking enabled")

2026-06-02 11:20:29,744 [INFO] Agent configured — model: llama-3.3-70b-versatile | memory tracking enabled


In [4]:
# Short Term Buffer

class ShortTermBuffer:
    def __init__(self, limit: int = SHORT_TERM_LIMIT):
        self.buffer: deque = deque(maxlen=limit)
        self.limit = limit

    def add(self, role: str, content: str) -> None:
        self.buffer.append({"role": role, "content": content})

    def get(self) -> List[Dict[str, str]]:
        return list(self.buffer)

    def clear(self) -> None:
        self.buffer.clear()
        log.info("Short term buffer cleared")

    def summary(self) -> str:
        return f"Short term: {len(self.buffer)}/{self.limit} messages"

short_term = ShortTermBuffer()
log.info(f"Short term buffer ready — capacity: {SHORT_TERM_LIMIT} messages")

2026-06-02 11:21:23,222 [INFO] Short term buffer ready — capacity: 10 messages


In [5]:
# Long Term Memory

class LongTermMemory:
    def __init__(self, path: str = CHROMA_PATH):
        Path(path).mkdir(parents=True, exist_ok=True)
        self.client = chromadb.PersistentClient(path=path)
        self.ef = embedding_functions.DefaultEmbeddingFunction()
        self.collection = self.client.get_or_create_collection(
            name="agent_memory",
            embedding_function=self.ef
        )
        log.info(f"Long term memory ready — stored entries: {self.collection.count()}")

    def store(self, memory_id: str, text: str, metadata: Dict) -> None:
        self.collection.upsert(
            ids=[memory_id],
            documents=[text],
            metadatas=[metadata]
        )

    def query(self, query_text: str, n_results: int = MEMORY_RESULTS) -> List[str]:
        count = self.collection.count()
        if count == 0:
            return []
        n = min(n_results, count)
        results = self.collection.query(query_texts=[query_text], n_results=n)
        return results["documents"][0] if results["documents"] else []

    def clear(self) -> None:
        self.client.delete_collection("agent_memory")
        self.collection = self.client.get_or_create_collection(
            name="agent_memory",
            embedding_function=self.ef
        )
        log.info("Long term memory cleared")

    def count(self) -> int:
        return self.collection.count()

long_term = LongTermMemory()

2026-06-02 11:21:45,562 [INFO] Long term memory ready — stored entries: 0


In [6]:
# Memory Manager

class MemoryManager:
    def __init__(self, short_term: ShortTermBuffer, long_term: LongTermMemory):
        self.short_term = short_term
        self.long_term = long_term
        self._turn_counter = 0

    def add_exchange(self, user_input: str, assistant_reply: str) -> None:
        self.short_term.add("user", user_input)
        self.short_term.add("assistant", assistant_reply)
        self._turn_counter += 1
        memory_id = f"turn_{self._turn_counter}_{int(time.time())}"
        self.long_term.store(
            memory_id=memory_id,
            text=f"User: {user_input}\nAssistant: {assistant_reply}",
            metadata={"turn": self._turn_counter, "timestamp": int(time.time())}
        )

    def retrieve_context(self, query: str) -> str:
        results = self.long_term.query(query)
        if not results:
            return ""
        context = "\n---\n".join(results)
        return f"Relevant memory context:\n{context}"

    def clear_all(self) -> None:
        self.short_term.clear()
        self.long_term.clear()
        self._turn_counter = 0
        log.info("All memory cleared")

    def summary(self) -> str:
        return f"{self.short_term.summary()} | Long term: {self.long_term.count()} entries | Turns: {self._turn_counter}"

memory = MemoryManager(short_term, long_term)
log.info("Memory manager ready")

2026-06-02 11:22:06,230 [INFO] Memory manager ready


In [7]:
# Chat Engine

def chat(user_input: str, mem: MemoryManager, cfg: AgentConfig) -> Optional[str]:
    retrieved_context = mem.retrieve_context(user_input)
    if retrieved_context:
        cfg.memory_hits += 1
        log.info(f"Memory hit #{cfg.memory_hits} — context retrieved")

    augmented_system = cfg.system_prompt
    if retrieved_context:
        augmented_system += f"\n\n{retrieved_context}"

    messages = [{"role": "system", "content": augmented_system}] + mem.short_term.get()
    messages.append({"role": "user", "content": user_input})

    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=cfg.model,
                messages=messages,
                max_tokens=cfg.max_tokens,
                temperature=cfg.temperature
            )
            reply = response.choices[0].message.content
            mem.add_exchange(user_input, reply)
            tokens = response.usage.total_tokens
            cfg.session_token_count += tokens
            log.info(f"Tokens this turn: {tokens} | Session total: {cfg.session_token_count}")
            return reply

        except Exception as e:
            log.warning(f"Attempt {attempt + 1} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)

    log.error("All retry attempts failed")
    return None

In [8]:
# Interactive Chat Loop

print("Agent ready. Commands: 'exit' to quit | 'clear' to reset memory | 'memory' for stats | 'recall <query>' to search memory\n")

while True:
    user_input = input("You: ").strip()

    if not user_input:
        continue
    if user_input.lower() == "exit":
        print(f"Session ended. Tokens: {config.session_token_count} | Memory hits: {config.memory_hits}")
        break
    if user_input.lower() == "clear":
        memory.clear_all()
        print("All memory cleared.\n")
        continue
    if user_input.lower() == "memory":
        print(f"{memory.summary()}\n")
        continue
    if user_input.lower().startswith("recall "):
        query = user_input[7:].strip()
        results = memory.long_term.query(query)
        print(f"Retrieved: {results}\n") if results else print("No matching memory found.\n")
        continue

    reply = chat(user_input, memory, config)
    print(f"\nAgent: {reply}\n")

Agent ready. Commands: 'exit' to quit | 'clear' to reset memory | 'memory' for stats | 'recall <query>' to search memory



You:  who are you


2026-06-02 11:23:00,049 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 11:23:02,205 [INFO] HTTP Request: GET https://chroma-onnx-models.s3.amazonaws.com/all-MiniLM-L6-v2/onnx.tar.gz "HTTP/1.1 200 OK"
C:\Users\USER\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz:  96%|██████▋| 76.0M/79.3M [04:54<00:13, 271kiB/s]
2026-06-02 11:27:56,398 [WARNING] Attempt 1 failed: The read operation timed out in upsert.
2026-06-02 11:27:59,629 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-02 11:28:02,048 [INFO] HTTP Request: GET https://chroma-onnx-models.s3.amazonaws.com/all-MiniLM-L6-v2/onnx.tar.gz "HTTP/1.1 200 OK"
C:\Users\USER\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|███████| 79.3M/79.3M [06:14<00:00, 222kiB/s]
2026-06-02 11:34:19,695 [INFO] Tokens this turn: 152 | Session total: 152



Agent: I am an advanced autonomous AI agent, utilizing a two-layer memory system for efficient information processing and recall. My primary function is to provide accurate and concise responses to a wide range of inquiries, leveraging my semantic recall capabilities and adapting to context provided in our exchange.



You:  my name is Charan and I am building a capstone project


...

Short term: 8/10 messages | Long term: 3 entries | Turns: 4



You:  recall capstone project


Retrieved: ["User: my name is Charan and I am building a capstone project\nAssistant: Hello Charan, I've taken note of your introduction. You're currently working on a capstone project. What kind of project is it, and how can I assist you with it?", "User: what do you know about me\nAssistant: I know that your name is Charan and you are currently working on a capstone project. That's the information you've shared with me so far.", 'User: who are you\nAssistant: I am an advanced autonomous AI agent, utilizing a two-layer memory system for efficient information processing and recall. My primary function is to provide accurate and concise responses to a wide range of inquiries, leveraging my semantic recall capabilities and adapting to context provided in our exchange.']



You:  exit


Session ended. Tokens: 936 | Memory hits: 2
